# Cost Decomposition Analysis

Decompose route costs into additive components:
- **cost_calm**: Hull resistance (depends on speed through water)
- **cost_waves**: Wave added resistance (depends on speed through water and wave height)
- **cost_wind**: Wind resistance (depends on speed through wind)

These sum exactly: `cost_total = cost_calm + cost_waves + cost_wind`

Current effects are isolated by comparing kinematics with vs without currents.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
gdf = gpd.read_parquet("../results/best_elites_decomposed_exact.geoparquet")

gdf["month"] = pd.to_datetime(gdf.journey_time_start).dt.month_name().str[:3]
gdf["direction"] = gdf.journey_name.map(
    {"Atlantic_forward": "eastward", "Atlantic_backward": "westward"}
)

print(f"Routes: {len(gdf)}")
print(f"Speeds: {sorted(gdf.journey_speed_knots.unique())}")
print(f"Directions: {gdf.direction.unique().tolist()}")

In [ ]:
# Verify decomposition sums exactly
sum_check = gdf.cost_calm + gdf.cost_waves + gdf.cost_wind
max_error = (sum_check - gdf.cost_total).abs().max()
print(f"Decomposition verification: max error = {max_error:.2e}")

# Hazard diagnosis
print(f"\nHazardous routes: {gdf.is_hazardous.sum()}/{len(gdf)}")
print(f"Max wave heights: {gdf.max_wave_height_m.min():.1f} - {gdf.max_wave_height_m.max():.1f} m")
print("\nHazardous by month:")
print(gdf.groupby("month").is_hazardous.agg(["sum", "count"]))

## Component Fractions by Speed and Direction

In [ ]:
fractions = gdf.groupby(["direction", "journey_speed_knots"]).agg(
    {
        "cost_calm": "mean",
        "cost_waves": "mean",
        "cost_wind": "mean",
        "cost_total": "mean",
    }
)

for col in ["cost_calm", "cost_waves", "cost_wind"]:
    fractions[f"{col}_pct"] = fractions[col] / fractions["cost_total"] * 100

fractions[["cost_calm_pct", "cost_waves_pct", "cost_wind_pct"]].round(1)

## Cost Components by Month, Speed, and Direction

In [ ]:
month_order = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

fig, axes = plt.subplots(2, 3, sharey=True)

for row, direction in enumerate(["eastward", "westward"]):
    for col, speed in enumerate([8, 10, 12]):
        ax = axes[row, col]
        df_sub = gdf[(gdf.journey_speed_knots == speed) & (gdf.direction == direction)]
        df_agg = (
            df_sub.groupby("month")[["cost_calm", "cost_waves", "cost_wind"]].mean()
            / 1e12
        )
        df_agg = df_agg.reindex(month_order)
        df_agg.plot(kind="bar", stacked=True, ax=ax, legend=(row == 0 and col == 2))
        
        # Mark hazardous months with red background
        hazard_by_month = df_sub.groupby("month").is_hazardous.any().reindex(month_order)
        for i, (month, is_haz) in enumerate(hazard_by_month.items()):
            if is_haz:
                ax.axvspan(i - 0.5, i + 0.5, color="red", alpha=0.15, zorder=0)
        
        ax.set_xticklabels(month_order)
        if row == 0:
            ax.set_title(f"{speed} kn")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=90)
        if col == 0:
            ax.set_ylabel(f"{direction}\nCost (TJ)")

fig.text(0.99, 0.01, "red = hazardous (H_s > L/40)", ha="right", fontsize=8, color="red")
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig("../figures/022_cost_components_stacked.png")
plt.savefig("../figures/022_cost_components_stacked.pdf")

## Current Effects by Direction

In [ ]:
current_effects = gdf.groupby(["direction", "journey_speed_knots"]).agg(
    {
        "delta_current_on_calm": "mean",
        "delta_current_on_waves": "mean",
        "delta_current_total": "mean",
        "cost_calm": "mean",
        "cost_total": "mean",
    }
)

current_effects["delta_calm_pct"] = (
    current_effects["delta_current_on_calm"] / current_effects["cost_calm"] * 100
)
current_effects["delta_total_pct"] = (
    current_effects["delta_current_total"] / current_effects["cost_total"] * 100
)

current_effects[["delta_calm_pct", "delta_total_pct"]].round(1)

In [ ]:
fig, axes = plt.subplots(2, 3, sharey="row")  # Share y only within rows

for row, direction in enumerate(["eastward", "westward"]):
    for col, speed in enumerate([8, 10, 12]):
        ax = axes[row, col]
        df_sub = gdf[(gdf.journey_speed_knots == speed) & (gdf.direction == direction)]
        df_agg = (
            df_sub.groupby("month")[
                ["delta_current_on_calm", "delta_current_on_waves"]
            ].mean()
            / 1e12
        )
        df_agg = df_agg.reindex(month_order)
        df_agg.columns = ["calm", "waves"]
        df_agg.plot(kind="bar", ax=ax, legend=(row == 0 and col == 2))
        ax.axhline(0, color="gray", linestyle="--")
        
        # Mark hazardous months with red background
        hazard_by_month = df_sub.groupby("month").is_hazardous.any().reindex(month_order)
        for i, (month, is_haz) in enumerate(hazard_by_month.items()):
            if is_haz:
                ax.axvspan(i - 0.5, i + 0.5, color="red", alpha=0.15, zorder=0)
        
        ax.set_xticklabels(month_order)
        if row == 0:
            ax.set_title(f"{speed} kn")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=90)
        if col == 0:
            ax.set_ylabel(f"{direction}\nCurrent effect (TJ)")

fig.text(0.99, 0.01, "red = hazardous (H_s > L/40)", ha="right", fontsize=8, color="red")
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig("../figures/022_current_effects.png")
plt.savefig("../figures/022_current_effects.pdf")

## Summary Table: Current Effect on Calm Water Cost (%)

In [ ]:
summary_rows = []

for direction in ["eastward", "westward"]:
    for speed in [8, 10, 12]:
        df_sub = gdf[(gdf.journey_speed_knots == speed) & (gdf.direction == direction)]
        df_agg = df_sub.groupby("month")[
            ["cost_calm", "delta_current_on_calm", "is_hazardous", "max_wave_height_m"]
        ].agg({
            "cost_calm": "mean",
            "delta_current_on_calm": "mean",
            "is_hazardous": "any",
            "max_wave_height_m": "max",
        })
        df_agg = df_agg.reindex(month_order)
        df_agg["pct"] = df_agg["delta_current_on_calm"] / df_agg["cost_calm"] * 100
        df_agg["speed"] = speed
        df_agg["direction"] = direction
        summary_rows.append(df_agg.reset_index())

summary_df = pd.concat(summary_rows)

for direction in ["eastward", "westward"]:
    print(f"\n{direction.upper()}")
    table = (
        summary_df[summary_df.direction == direction]
        .pivot(index="month", columns="speed", values="pct")
        .reindex(month_order)
    )
    table.columns = [f"{s} kn" for s in table.columns]
    
    # Add hazard column
    hazard_table = (
        summary_df[summary_df.direction == direction]
        .pivot(index="month", columns="speed", values="is_hazardous")
        .reindex(month_order)
    )
    table["hazard"] = hazard_table.any(axis=1).map({True: "hazardous", False: ""})
    
    print(table.round(1).to_string())

## Summary

In [ ]:
summary_df.to_csv("../results/022_cost_decomposition_summary.csv", index=False)
print("Saved to ../results/022_cost_decomposition_summary.csv")

# Summary of non-hazardous routes
gdf_safe = gdf[~gdf.is_hazardous]
print(f"\n=== Non-hazardous routes: {len(gdf_safe)}/{len(gdf)} ===")
if len(gdf_safe) > 0:
    safe_effects = gdf_safe.groupby("direction").agg({
        "delta_current_total": "mean",
        "cost_total": "mean",
    })
    safe_effects["pct"] = safe_effects["delta_current_total"] / safe_effects["cost_total"] * 100
    print(safe_effects[["pct"]].round(1))